# Water Accreted Axis: Atmosphere-Only Example

This notebook shows the workflow in three executable cells: run the model, load/process the output table, and plot atmosphere species.

In [ ]:
# Cell 1: run a small sweep over accreted water.
from pathlib import Path
import sys

import numpy as np

# Make imports work whether the notebook is launched from the repo root or from Example/.
repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from Example.gce_organizer import GCEParams
from Example.run_gce import run_gce

run_dir = run_gce(
    run_name="notebook_examples/water_accreted_axis",
    params=GCEParams(
        tarWaterarray=np.array([0.00, 0.02, 0.04, 0.06, 0.08, 0.10]),
        tarHHearray=np.array([0.03]),
    ),
    version="Sulfur_Nitrogen_Version",
    axis_list=["Water"],
    verbose=False,
)

run_dir = Path(run_dir)
run_dir

In [ ]:
# Cell 2: read results.dat and reshape atmosphere species for plotting.
import pandas as pd

from Example.plots.helpers.plotting_helpers import axis_label, axis_series

results_path = run_dir / "results.dat"
df = pd.read_csv(results_path, sep=r"\s+")
df = df.rename(columns={"#index": "index"}).sort_values("fWater").reset_index(drop=True)

water_wt_percent = axis_series(df, "Water")
atmosphere_species = [
    "H2O_gas",
    "H2_gas",
    "CO2_gas",
    "CO_gas",
    "CH4_gas",
    "O2_gas",
    "N2_gas",
    "NH3_gas",
    "S2_gas",
    "SO2_gas",
    "H2S_gas",
]
atmosphere_species = [name for name in atmosphere_species if name in df.columns]

atm = df.loc[:, atmosphere_species].apply(pd.to_numeric, errors="coerce")
atm = atm.where(atm > 0.0)
atm.insert(0, "water_wt_percent", water_wt_percent)

atm_long = atm.melt(
    id_vars="water_wt_percent",
    var_name="species",
    value_name="mole_fraction",
).dropna()

atm_long.head()

In [ ]:
# Cell 3: plot atmosphere-only mole fractions along the water accreted axis.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4.5))

for species, subset in atm_long.groupby("species", sort=False):
    ax.plot(
        subset["water_wt_percent"],
        subset["mole_fraction"],
        marker="o",
        linewidth=1.8,
        label=species.replace("_gas", ""),
    )

ax.set_yscale("log")
ax.set_xlabel(axis_label("Water"))
ax.set_ylabel("Atmosphere mole fraction")
ax.set_title("Atmosphere species vs. accreted water")
ax.grid(True, which="both", linestyle=":", alpha=0.45)
ax.legend(title="Species", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
fig.tight_layout()
plt.show()